# **PROJECT**
# **Comment Category Prediction Challenge :**

# Imports Libraries

In [ ]:
# Imports

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import gc
import psutil
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

# RAM checker
def ram():
    v = psutil.virtual_memory()
    print(f'RAM  used={v.used/1e9:.1f}GB  free={v.available/1e9:.1f}GB  total={v.total/1e9:.1f}GB')


# Data Loading

In [ ]:
# AUTO DETECT PATH

base_path = '/kaggle/input'

all_files = []
for dirname, _, filenames in os.walk(base_path):
    for filename in filenames:
        full_path = os.path.join(dirname, filename)
        all_files.append(full_path)

train_path  = [f for f in all_files if 'train.csv'  in f.lower()][0]
test_path   = [f for f in all_files if 'test.csv'   in f.lower()][0]
sample_path = [f for f in all_files if 'sample'     in f.lower()][0]

print("Detected Paths:")
print(train_path)
print(test_path)
print(sample_path)

train  = pd.read_csv(train_path)
test   = pd.read_csv(test_path)
sample = pd.read_csv(sample_path)

print("\nTrain shape:", train.shape)
print("Test shape:", test.shape)
print("Sample shape:", sample.shape)


# Data Cleaning

In [ ]:
# Data Cleaning

for df in [train, test]:
    df['created_date'] = pd.to_datetime(df['created_date'], errors='coerce', utc=True)
    for col in ['race', 'religion', 'gender']:
        df[col] = df[col].fillna('unknown')
    df['comment']    = df['comment'].fillna('')
    df['disability'] = df['disability'].fillna(0).astype(int)
    for col in ['upvote', 'downvote', 'if_1', 'if_2',
                'emoticon_1', 'emoticon_2', 'emoticon_3']:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

print('Nulls remaining:', train.isnull().sum().sum())
print('All clean!' if train.isnull().sum().sum() == 0 else 'WARNING: nulls remain')


# Text Cleaning

In [ ]:
# Text Cleaning

def clean_comment(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', ' url ', text)
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'[^a-z0-9\s!?]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train['clean_comment'] = train['comment'].apply(clean_comment)
test['clean_comment']  = test['comment'].apply(clean_comment)

print('Text cleaning done.')
print('Sample:', train['clean_comment'].iloc[0][:100])


# **Feature Engineering** :

In [ ]:
# Feature Engineering

import gc

for df in [train, test]:

    # Datetime
    df['year']       = df['created_date'].dt.year
    df['month']      = df['created_date'].dt.month
    df['day']        = df['created_date'].dt.day
    df['hour']       = df['created_date'].dt.hour
    df['dayofweek']  = df['created_date'].dt.dayofweek
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)

    # Text stats
    df['comment_length']    = df['comment'].apply(len)
    df['word_count']        = df['comment'].apply(lambda x: len(x.split()))
    df['avg_word_length']   = df['comment_length'] / (df['word_count'] + 1)
    df['exclamation_count'] = df['comment'].str.count('!')
    df['question_count']    = df['comment'].str.count(r'\?')
    df['uppercase_ratio']   = df['comment'].apply(
        lambda x: sum(1 for c in x if c.isupper()) / (len(x) + 1))
    df['unique_words_ratio'] = df['clean_comment'].apply(
        lambda x: len(set(x.split())) / (len(x.split()) + 1) if x.split() else 0)
    df['punctuation_count'] = df['comment'].str.count(r'[!?.,;:]')
    df['punctuation_ratio'] = df['punctuation_count'] / (df['comment_length'] + 1)
    df['url_count']         = df['comment'].str.count(r'http\S+|www\S+')
    df['sentence_count']    = df['comment'].str.count(r'[.!?]') + 1
    df['avg_sentence_len']  = df['word_count'] / (df['sentence_count'] + 1)
    df['cap_word_count']    = df['comment'].apply(
        lambda x: sum(1 for w in x.split() if w.isupper() and len(w) > 1))
    df['repeat_char']       = df['comment'].apply(
        lambda x: len(re.findall(r'(.)\1{2,}', x)))
    df['toxic_punc']        = df['comment'].str.count(r'[!?]{2,}')
    df['has_url']           = (df['url_count'] > 0).astype(int)

    # Vote features
    df['vote_ratio']     = df['upvote'] / (df['downvote'] + 1)
    df['total_votes']    = df['upvote'] + df['downvote']
    df['vote_sentiment'] = df['upvote'] - df['downvote']
    df['log_upvote']     = np.log1p(df['upvote'])
    df['log_downvote']   = np.log1p(df['downvote'])
    df['vote_diff_log']  = np.log1p(df['upvote']) - np.log1p(df['downvote'])

    # Emoticon features
    df['total_emoticons']  = df['emoticon_1'] + df['emoticon_2'] + df['emoticon_3']
    df['has_any_emoticon'] = (df['total_emoticons'] > 0).astype(int)
    df['emoticon_ratio']   = df['total_emoticons'] / (df['word_count'] + 1)

    # if_2 bucket features
    df['if_2_is_10'] = (df['if_2'] == 10).astype(int)
    df['if_2_is_4']  = (df['if_2'] == 4).astype(int)
    df['if_2_is_11'] = (df['if_2'] == 11).astype(int)

    df['if_2_bucket'] = pd.cut(
        df['if_2'],
        bins=[-1, 0, 4, 9, 14, 999],
        labels=False
    ).fillna(-1).astype(int)

    # Interaction features
    df['if1_if2_product'] = df['if_1'] * df['if_2']
    df['if1_x_dis']       = df['if_1'] * df['disability']
    df['if2_x_downvote']  = df['if_2'] * df['downvote']
    df['if1_x_upvote']    = df['if_1'] * df['upvote']
    df['if2_x_len']       = df['if_2'] * df['comment_length']
    df['downvote_x_len']  = df['downvote'] * df['comment_length']

# Drop created_date
train.drop(columns=['created_date'], inplace=True)
test.drop(columns=['created_date'], inplace=True)

gc.collect()

print('Feature engineering done. Train shape:', train.shape)


#  Post Features

In [ ]:
# Post-Level Aggregation + Bayesian Target Encoding

def add_post_features(df, reference_df=None):
    src = reference_df if reference_df is not None else df
    post_stats = src.groupby('post_id').agg(
        post_upvote_mean   = ('upvote',   'mean'),
        post_upvote_std    = ('upvote',   'std'),
        post_upvote_max    = ('upvote',   'max'),
        post_downvote_mean = ('downvote', 'mean'),
        post_downvote_std  = ('downvote', 'std'),
        post_downvote_max  = ('downvote', 'max'),
        post_comment_count = ('upvote',   'count')
    ).reset_index()
    if 'label' in src.columns:
        global_label_mean = src['label'].mean()
        k = 5
        label_agg = src.groupby('post_id')['label'].agg(
            _sum='sum', _count='count').reset_index()
        label_agg['post_label_mean'] = (
            (label_agg['_sum'] + global_label_mean * k)
            / (label_agg['_count'] + k))
        label_agg['post_label_std'] = \
            src.groupby('post_id')['label'].std().fillna(0).values
        label_agg = label_agg[['post_id', 'post_label_mean', 'post_label_std']]
        post_stats = post_stats.merge(label_agg, on='post_id', how='left')
    df = df.merge(post_stats, on='post_id', how='left')
    return df

train = add_post_features(train)
test  = add_post_features(test, reference_df=train)

post_cols = [c for c in train.columns if c.startswith('post_')]
train[post_cols] = train[post_cols].fillna(0)
test[post_cols]  = test[post_cols].fillna(0)

gc.collect()
print('Post features added:', post_cols)
print('Train shape:', train.shape)


# TF-IDF Vectorization

In [ ]:
# TF-IDF Vectorization

from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

tfidf_word = TfidfVectorizer(
    max_features  = 50000,
    ngram_range   = (1, 2),
    min_df        = 3,
    max_df        = 0.95,
    sublinear_tf  = True,
    strip_accents = 'unicode'
)

tfidf_char = TfidfVectorizer(
    max_features  = 30000,
    analyzer      = 'char_wb',
    ngram_range   = (3, 4),
    min_df        = 10,
    max_df        = 0.95,
    sublinear_tf  = True,
    strip_accents = 'unicode'
)

print('Fitting word TF-IDF...')
X_word_train = tfidf_word.fit_transform(train['clean_comment'])
X_word_test  = tfidf_word.transform(test['clean_comment'])
print('Word TF-IDF:', X_word_train.shape)
gc.collect()

print('Fitting char TF-IDF...')
X_char_train = tfidf_char.fit_transform(train['clean_comment'])
X_char_test  = tfidf_char.transform(test['clean_comment'])
print('Char TF-IDF:', X_char_train.shape)
gc.collect()
ram()


# Sparse Matrix

In [ ]:
# Encode Categoricals & Build Final Sparse Matrix

from sklearn.preprocessing import LabelEncoder

cat_cols = ['race', 'religion', 'gender']
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train[col].astype(str), test[col].astype(str)])
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))

meta_features = [
    'emoticon_1', 'emoticon_2', 'emoticon_3',
    'upvote', 'downvote', 'if_1', 'if_2',
    'race', 'religion', 'gender', 'disability',
    'year', 'month', 'day', 'hour', 'dayofweek', 'is_weekend',
    'comment_length', 'word_count', 'avg_word_length',
    'exclamation_count', 'question_count', 'uppercase_ratio',
    'unique_words_ratio', 'punctuation_count', 'punctuation_ratio',
    'url_count', 'sentence_count', 'avg_sentence_len',
    'cap_word_count', 'repeat_char', 'toxic_punc', 'has_url',
    'vote_ratio', 'total_votes', 'vote_sentiment',
    'log_upvote', 'log_downvote', 'vote_diff_log',
    'total_emoticons', 'has_any_emoticon', 'emoticon_ratio',
    'if_2_is_10', 'if_2_is_4', 'if_2_is_11', 'if_2_bucket',
    'if1_if2_product', 'if1_x_dis', 'if2_x_downvote',
    'if1_x_upvote', 'if2_x_len', 'downvote_x_len',
] + post_cols

print(f'Total meta features: {len(meta_features)}')

train[meta_features] = train[meta_features].fillna(0).astype(float)
test[meta_features]  = test[meta_features].fillna(0).astype(float)

X_meta_train = csr_matrix(train[meta_features].values)
X_meta_test  = csr_matrix(test[meta_features].values)

X_train_full = hstack([X_word_train, X_char_train, X_meta_train], format='csr')
X_test_full  = hstack([X_word_test,  X_char_test,  X_meta_test],  format='csr')

del X_word_train, X_char_train, X_meta_train
del X_word_test,  X_char_test,  X_meta_test
gc.collect()

y = train['label']

print('Final Train Matrix:', X_train_full.shape)
print('Final Test  Matrix:', X_test_full.shape)
ram()


#  Validation Split

In [ ]:
# Stratified Train / Validation Split

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_full, y,
    test_size    = 0.15,
    random_state = 42,
    stratify     = y
)

gc.collect()
print('Train :', X_tr.shape, '| Val:', X_val.shape)
print('\nVal label distribution:')
print(y_val.value_counts().sort_index())
ram()


# **Modeling — LightGBM** :

In [ ]:
# MODEL : LightGBM

import lightgbm as lgb
import gc
from sklearn.metrics import f1_score, classification_report

gc.collect()
print('Training LightGBM...')

lgb_model1 = lgb.LGBMClassifier(
    n_estimators      = 1500,
    learning_rate     = 0.05,
    num_leaves        = 127,
    max_depth         = -1,
    min_child_samples = 20,
    subsample         = 0.8,
    subsample_freq    = 1,
    colsample_bytree  = 0.6,
    reg_alpha         = 0.1,
    reg_lambda        = 0.1,
    class_weight      = 'balanced',
    objective         = 'multiclass',
    n_jobs            = -1,
    random_state      = 42,
    verbose           = -1
)
lgb_model1.fit(
    X_tr, y_tr,
    eval_set  = [(X_val, y_val)],
    callbacks = [
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(period=50)
    ]
)

y_pred_lgb1   = lgb_model1.predict(X_val)
lgb1_macro_f1 = f1_score(y_val, y_pred_lgb1, average='macro')

print(f'\nLightGBM  Macro F1 : {lgb1_macro_f1:.4f}   <- Kaggle metric')
print('\nPer-class breakdown:')
print(classification_report(y_val, y_pred_lgb1, digits=4))
gc.collect()
ram()

final_model = lgb_model1
final_macro_f1 = lgb1_macro_f1
X_test_final = X_test_full


# Test Predictions

In [ ]:
# Test Predictions

print('Generating predictions...')

test_predictions = final_model.predict(X_test_final)

print('Test distribution:')
print(pd.Series(test_predictions).value_counts().sort_index())
print('\nTrain distribution (reference):')
print(y.value_counts().sort_index())


# Submission File

In [ ]:
# Save Submission

submission = pd.DataFrame({
    'ID'   : sample['ID'],
    'label': test_predictions
})
submission.to_csv('submission.csv', index=False)

print('submission.csv saved!')
print(f'Rows         : {len(submission)}')
print(f'Val Macro F1 : {final_macro_f1:.4f}  <- leaderboard estimate')
display(submission.head(10))
